# Pancreas — Multi-Seed Confirmation: baseline vs Fresnel

Confirms whether the **Fresnel mass effect** (large +Dice on the hard tumour class, seen in the single run) is real and robust, not an initialization fluke. Each arm is trained under **multiple seeds** (`cfg.seeds`), each with a different model init and train/val split, all evaluated on a **fixed held-out test set** (`cfg.test_seed`) so per-case pairing is clean.

**Stats:** the PRIMARY test averages each test case's Dice across seeds, then runs a paired Wilcoxon vs baseline per class, with **Holm correction**. Per-seed sign-consistency is reported alongside. A `*SIG*` on the mass row that survives Holm = a defensible positive result.

**Resumable:** per-(arch,seed) checkpoints; a dropped session resumes with `cfg.retrain=False`. `edge_att` is omitted here to keep the seed sweep tractable. Use a GPU runtime; raw via local disk (`DOWNLOAD_LOCAL=True`) if the cache isn't built. Outputs → `Outputs/Pancreas-Compare/`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
import numpy as np, torch

DRIVE = Path("/content/drive/MyDrive")

@dataclass
class Config:
    # ---------------- DATA ----------------
    raw_root: Path = DRIVE/"Datasets"/"Pancreas"          # full-res CT: imagesTr/ + labelsTr/
    cache_root: Path = DRIVE/"Datasets"/"PancreasCache"   # compact cache written/read here
    rebuild_cache: bool = False                           # auto-rebuilds if target_shape/HU changes
    retrain: bool = True                                 # THIS run: retrain all. Set False to RESUME after a drop.
    early_stop_patience: int = 12                        # stop an arm if val Dice doesn't improve for N epochs (0=off)
    hu_window: tuple = (-100.0, 240.0)                    # abdominal soft-tissue window (HU)
    target_shape: tuple = (112, 112, 112)                    # raise to (112,)*3 / (128,)*3 to resolve the mass
    n_cases: int = 0                                      # 0/None = all; else random seeded subset
    val_fraction: float = 0.15
    test_fraction: float = 0.15

    # ---------------- MODEL / COMPARISON ----------------
    arches: tuple = ("baseline", "fresnel")   # focus: confirm the Fresnel mass effect (add "sea" if desired)
    base_channels: int = 16
    n_bands: int = 4
    n_dist: int = 3

    # ---------------- TRAINING ----------------
    epochs: int = 40
    batch_size: int = 1
    lr: float = 2e-3
    weight_decay: float = 1e-5
    w_dice: float = 1.0; w_ce: float = 1.0
    amp: bool = True

    # ---------------- EVALUATION ----------------
    robustness: bool = True
    noise_sigmas: tuple = (0.0, 0.05, 0.10, 0.20)
    blur_sigmas: tuple = (0.0, 0.5, 1.0)

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seeds: tuple = (0, 1, 2, 3, 4)        # multi-seed: train each arm under each seed
    test_seed: int = 12345               # FIXED held-out test split (same 42 cases every seed)
    seed: int = 0
    out_base: Path = DRIVE/"Outputs"/"Pancreas-Compare"

cfg = Config()
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
OUT = cfg.out_base; FIG, TBL, CKPT = OUT/"Figures", OUT/"Tables", OUT/"Checkpoints"
SUMMARY = OUT/"outputs_summary.txt"
for d in (OUT, FIG, TBL, CKPT): d.mkdir(parents=True, exist_ok=True)
print("device:", cfg.device, "| arches:", cfg.arches, "| target:", cfg.target_shape, "| outputs ->", OUT)


## 1b. (Optional) Download raw data to **local Colab disk**
Avoids using Drive storage/quota for the 12 GB of raw CT. Set `DOWNLOAD_LOCAL=True` to use it; then it sets `cfg.raw_root` to the local copy. **Skip this entirely if your cache is already built** (the cache is all training needs).

In [ ]:
DOWNLOAD_LOCAL = False   # set True to fetch raw CT to local session disk instead of reading from Drive
if DOWNLOAD_LOCAL:
    import os
    os.makedirs("/content/pancreas_data", exist_ok=True)
    if not os.path.exists("/content/pancreas_data/Task07_Pancreas"):
        print("downloading Task07_Pancreas.tar (~11.5 GB) to local disk ...")
        !wget -q --show-progress https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar -O /content/pancreas_data/Task07_Pancreas.tar
        !tar -xf /content/pancreas_data/Task07_Pancreas.tar -C /content/pancreas_data/
        !rm /content/pancreas_data/Task07_Pancreas.tar
    from pathlib import Path
    cfg.raw_root = Path("/content/pancreas_data/Task07_Pancreas")   # build cache from local disk
    print("raw_root ->", cfg.raw_root)
else:
    print("DOWNLOAD_LOCAL=False — using cfg.raw_root as configured")


## 3. Data — NIfTI, robust path/name matching, automatic class detection

In [ ]:
import numpy as np, torch, json
import nibabel as nib, torch.nn.functional as F, re

# ---------- raw listing (robust) ----------
def _ci_find(parent, names):
    if not parent.exists(): return None
    kids={p.name.lower():p for p in parent.iterdir() if p.is_dir()}
    for n in names:
        if n.lower() in kids: return kids[n.lower()]
    return None
def _nii(d): return [p for p in sorted(list(d.glob("*.nii.gz"))+list(d.glob("*.nii"))) if not p.name.startswith((".","._"))] if d and d.exists() else []
def _stem(n):
    for e in (".nii.gz",".nii"):
        if n.lower().endswith(e): n=n[:-len(e)]; break
    n=n.lower()
    for t in ("_processed","_seg","_label","_gt","_mask"): n=n.replace(t,"")
    return n.strip("_-")
def _num(n): m=re.findall(r"\d+",n); return (m[-1].lstrip("0") or "0") if m else None
def _pair(imgs,lbls):
    be={p.name:p for p in lbls}; bs={_stem(p.name):p for p in lbls}; bn={_num(p.name):p for p in lbls}
    out=[]
    for p in imgs:
        lp=be.get(p.name) or bs.get(_stem(p.name)) or bn.get(_num(p.name))
        if lp is not None: out.append((p,lp,_stem(p.name)))
    if not out and len(imgs)==len(lbls): out=[(p,l,_stem(p.name)) for p,l in zip(imgs,lbls)]
    return out

def _resize(v,s,m):
    t=torch.from_numpy(v)[None,None].float()
    return F.interpolate(t,size=s,mode=m,align_corners=False if m!="nearest" else None)[0,0].numpy()

# ---------- one-time cache build ----------
def build_cache(cfg):
    """Resumable, Drive-robust cache builder.
    - skips cases already cached (re-run after adding new images -> only new ones build)
    - copies each raw file to local disk before reading (avoids FUSE/Drive streaming drops)
    - retries on transient Drive disconnects
    - rebuild_cache=True or a target_shape/HU change forces a full rebuild
    """
    import time, shutil
    cfg.cache_root.mkdir(parents=True, exist_ok=True)
    meta_p=cfg.cache_root/"meta.json"
    force=cfg.rebuild_cache
    if meta_p.exists():
        meta=json.load(open(meta_p))
        if meta.get("target_shape")!=list(cfg.target_shape) or meta.get("hu_window")!=list(cfg.hu_window):
            print("  [cache] target_shape/HU changed -> full rebuild"); force=True
    idir=_ci_find(cfg.raw_root,["imagesTr","images","Images"]); ldir=_ci_find(cfg.raw_root,["labelsTr","labels","Labels"])
    cases=_pair(_nii(idir),_nii(ldir))
    if not cases: raise FileNotFoundError(f"No raw pairs under {cfg.raw_root} (looked for imagesTr/labelsTr).")
    if cfg.n_cases:                       # random seeded subset (deterministic)
        idx=np.arange(len(cases)); np.random.default_rng(cfg.seed).shuffle(idx)
        cases=[cases[i] for i in idx[:int(cfg.n_cases)]]
    lo,hi=cfg.hu_window; ts=cfg.target_shape
    tmp=Path("/content/_rawtmp"); 
    try: tmp.mkdir(exist_ok=True)
    except Exception: tmp=cfg.cache_root  # fallback if not on Colab

    def _read(path):                      # local-copy + retry, robust to Drive drops
        for attempt in range(5):
            try:
                local=tmp/Path(path).name
                shutil.copy(str(path), local)
                arr=np.asarray(nib.load(str(local)).get_fdata(),dtype=np.float32)
                try: local.unlink()
                except Exception: pass
                return arr
            except (OSError, ConnectionError) as e:
                print(f"      retry {attempt+1} on {Path(path).name}: {type(e).__name__}"); time.sleep(3)
        raise RuntimeError(f"failed to read {path} after retries")

    done=set() if force else {p.stem for p in cfg.cache_root.glob("*.npz")}
    todo=[c for c in cases if c[2] not in done]
    print(f"cache: {len(done)} already done, {len(todo)} to build -> {cfg.cache_root}  (HU{cfg.hu_window}, {ts})")
    for k,(ip,lp,cid) in enumerate(todo):
        img=_read(ip); lab=np.round(_read(lp)).astype(np.int64)
        if k==0:
            print(f"  [diag] native shape {img.shape} | HU 1/99 pct "
                  f"{round(float(np.percentile(img,1)))}/{round(float(np.percentile(img,99)))} | label values {np.unique(lab).tolist()}")
        img=np.clip(img,lo,hi); img=((img-lo)/(hi-lo)*255.0)
        img=_resize(img,ts,"trilinear").clip(0,255).astype(np.uint8)
        lab=_resize(lab.astype(np.float32),ts,"nearest").astype(np.uint8)
        np.savez_compressed(cfg.cache_root/f"{cid}.npz", img=img, lab=lab)
        if (k+1)%20==0: print(f"   cached {k+1}/{len(todo)}")
    n=len(list(cfg.cache_root.glob("*.npz")))
    json.dump(dict(target_shape=list(ts),hu_window=list(cfg.hu_window),n=n),open(meta_p,"w"))
    mb=sum(p.stat().st_size for p in cfg.cache_root.glob("*.npz"))/1e6
    print(f"cache complete: {n} cases, {mb:.0f} MB total")

build_cache(cfg)

# ---------- load from cache ----------
cache_files=sorted(p for p in cfg.cache_root.glob("*.npz"))
def load_cached(p):
    d=np.load(p); return d["img"].astype(np.float32), d["lab"].astype(np.int64)
def detect_classes_cache(files,sample=60):
    vals=set()
    for p in files[:sample]: vals.update(int(v) for v in np.unique(np.load(p)["lab"]) if v>0)
    fg=sorted(vals); return {v:i+1 for i,v in enumerate(fg)}, len(fg)
LMAP,NFG=detect_classes_cache(cache_files); NCLS=NFG+1
CLASS_NAMES={1:"pancreas",2:"mass"} if NFG==2 else {c:f"class{c}" for c in range(1,NCLS)}

class DS(torch.utils.data.Dataset):
    def __init__(s,cases,cfg,lmap,aug=False): s.c=cases; s.cfg=cfg; s.lmap=lmap; s.aug=aug
    def __len__(s): return len(s.c)
    def __getitem__(s,i):
        ip,lp,cid=s.c[i]                                      # ip==lp==npz path (cache)
        d=np.load(ip); img=d["img"].astype(np.float32); raw=d["lab"].astype(np.int64)
        img=(img-img.mean())/(img.std()+1e-6)                 # per-volume z-score
        lab=np.zeros_like(raw,dtype=np.int64)
        for v,k in s.lmap.items(): lab[raw==v]=k
        if s.aug and np.random.rand()<0.5: img=img+np.random.normal(0,0.05,img.shape).astype(np.float32)
        return torch.from_numpy(img.astype(np.float32))[None], torch.from_numpy(lab), cid

# fixed held-out TEST set (seed-independent) + train/val pool (split per-seed in the loop)
allidx=np.arange(len(cache_files)); np.random.default_rng(cfg.test_seed).shuffle(allidx)
ntest=int(len(cache_files)*cfg.test_fraction)
test_files=[cache_files[i] for i in allidx[:ntest]]
trainval_files=[cache_files[i] for i in allidx[ntest:]]
test_all=[(p,p,p.stem) for p in test_files]
def split_trainval(seed):
    ix=np.arange(len(trainval_files)); np.random.default_rng(seed).shuffle(ix)
    nv=max(1,int(len(trainval_files)*cfg.val_fraction))
    val=[trainval_files[i] for i in ix[:nv]]; tr=[trainval_files[i] for i in ix[nv:]]
    return [(p,p,p.stem) for p in tr],[(p,p,p.stem) for p in val]
print(f"classes: {NFG} fg -> {NCLS}-way {LMAP} ({'/'.join(CLASS_NAMES.get(c,str(c)) for c in range(1,NCLS))}) | "
      f"FIXED test={len(test_all)} | trainval pool={len(trainval_files)} | seeds={cfg.seeds}")


# cache-aware prep used by evaluation (ip==lp==npz path)
def prep(ip, lp, cfg, lmap):
    d=np.load(ip); img=d["img"].astype(np.float32); raw=d["lab"].astype(np.int64)
    img=(img-img.mean())/(img.std()+1e-6)
    lab=np.zeros_like(raw,dtype=np.int64)
    for v,k in lmap.items(): lab[raw==v]=k
    return img.astype(np.float32), lab


## 4. Models — shared backbone, swappable module

In [ ]:
import math, torch.nn as nn

class MultiBandSpectralEdgeAttention3D(nn.Module):
    def __init__(self, channels, n_bands=4):
        super().__init__()
        self.mu=nn.Parameter(torch.linspace(0.45,0.90,n_bands).clone())
        self.log_sigma=nn.Parameter(torch.log(torch.full((n_bands,),0.08)))
        self.fuse=nn.Conv3d(channels*n_bands,channels,1)
        self.gate=nn.Sequential(nn.Conv3d(channels,channels,1),nn.Sigmoid()); self.n=n_bands
    def _rho(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); r=torch.sqrt(Z**2+Y**2+X**2); return r/(r.max()+1e-6)
    def forward(s,x):
        D,H,W=x.shape[-3:]; Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1)); rho=s._rho(D,H,W,x.device)
        sg=torch.exp(s.log_sigma); mu=torch.clamp(s.mu,0,1)
        bands=[torch.fft.irfftn(Xf*torch.exp(-((rho-mu[b])**2)/(2*sg[b]**2+1e-6)).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for b in range(s.n)]
        return x*(1.0+s.gate(s.fuse(torch.cat(bands,1))))

class FresnelPropagation3D(nn.Module):
    def __init__(self, channels, n_dist=3):
        super().__init__()
        self.z=nn.Parameter(torch.linspace(0.05,0.30,n_dist)); self.fuse=nn.Conv3d(channels*n_dist,channels,1); self.n=n_dist
    def _k2(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); return Z**2+Y**2+X**2
    def forward(s,x):
        D,H,W=x.shape[-3:]; Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1)); k2=s._k2(D,H,W,x.device)
        outs=[torch.fft.irfftn(Xf*torch.exp(1j*(2*math.pi)*s.z[i]*k2).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for i in range(s.n)]
        return x+s.fuse(torch.cat(outs,1))

class ResBlock3D(nn.Module):
    def __init__(s,ci,co):
        super().__init__(); s.c1=nn.Conv3d(ci,co,3,padding=1); s.n1=nn.InstanceNorm3d(co)
        s.c2=nn.Conv3d(co,co,3,padding=1); s.n2=nn.InstanceNorm3d(co); s.skip=nn.Conv3d(ci,co,1) if ci!=co else nn.Identity()
    def forward(s,x): h=torch.relu(s.n1(s.c1(x))); h=s.n2(s.c2(h)); return torch.relu(h+s.skip(x))

def make_module(arch, ch, cfg):
    if arch=="sea": return MultiBandSpectralEdgeAttention3D(ch, cfg.n_bands)
    if arch=="fresnel": return FresnelPropagation3D(ch, cfg.n_dist)
    return nn.Identity()

class CompareUNet3D(nn.Module):
    def __init__(s, cfg, n_classes, arch="baseline", in_ch=1):
        super().__init__(); base=cfg.base_channels
        s.e1=ResBlock3D(in_ch,base); s.e2=ResBlock3D(base,2*base); s.bott=ResBlock3D(2*base,4*base)
        s.up2=nn.ConvTranspose3d(4*base,2*base,2,2); s.d2=ResBlock3D(4*base,2*base)
        s.up1=nn.ConvTranspose3d(2*base,base,2,2); s.d1=ResBlock3D(2*base,base)
        s.m2=make_module(arch,2*base,cfg); s.m1=make_module(arch,base,cfg)
        s.out=nn.Conv3d(base,n_classes,1); s.pool=nn.MaxPool3d(2)
    def forward(s,x):
        e1=s.e1(x); e2=s.e2(s.pool(e1)); b=s.bott(s.pool(e2))
        d2=s.m2(s.d2(torch.cat([s.up2(b),e2],1))); d1=s.m1(s.d1(torch.cat([s.up1(d2),e1],1)))
        return s.out(d1)


# ============================================================================
# Paper-1 reference model: edge_att  (edge-attention encoder + transformer bottleneck)
# Added as a 4th comparison arm. NOTE: this is a *different architecture* from the
# baseline/SEA/Fresnel backbone (4 pool levels, BatchNorm, dropout, ViT bottleneck) —
# it is the published reference model, not a module swap. SEA/Fresnel remain at the
# DECODER of the shared backbone (where spectral operators have resolution to act);
# they are deliberately NOT inserted into edge_att's 6^3 bottleneck.
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F

def _gn(c):
    g = 8 if c%8==0 else (4 if c%4==0 else 1)
    return nn.GroupNorm(num_groups=g, num_channels=c)

class _DoubleConv3D(nn.Module):
    def __init__(self,ci,co,dr=0.2):
        super().__init__(); self.conv=nn.Sequential(
            nn.Conv3d(ci,co,3,padding=1),_gn(co),nn.ReLU(inplace=True),nn.Dropout3d(dr),
            nn.Conv3d(co,co,3,padding=1),_gn(co),nn.ReLU(inplace=True),nn.Dropout3d(dr))
    def forward(s,x): return s.conv(x)
class _ResidualConvBlock3D(nn.Module):
    def __init__(s,ci,co,dr=0.2):
        super().__init__(); s.conv=nn.Sequential(
            nn.Conv3d(ci,co,3,padding=1,bias=False),_gn(co),nn.ReLU(inplace=True),nn.Dropout3d(dr),
            nn.Conv3d(co,co,3,padding=1,bias=False),_gn(co))
        s.shortcut=nn.Conv3d(ci,co,1,bias=False); s.relu=nn.ReLU(inplace=True)
    def forward(s,x): return s.relu(s.conv(x)+s.shortcut(x))
class _MultiScaleEdgeDetector3D(nn.Module):
    def __init__(s,in_channels):
        super().__init__()
        sx=torch.tensor([[[-1,0,1],[-2,0,2],[-1,0,1]]]*3,dtype=torch.float32)
        sy=torch.tensor([[[-1,-2,-1],[0,0,0],[1,2,1]]]*3,dtype=torch.float32)
        sz=torch.tensor([[[-1,-1,-1],[-2,-2,-2],[-1,-1,-1]],[[0,0,0]]*3,[[1,1,1],[2,2,2],[1,1,1]]],dtype=torch.float32)
        for n,k in (("sobel_x",sx),("sobel_y",sy),("sobel_z",sz)): s.register_buffer(n,k.unsqueeze(0).unsqueeze(0))
    def forward(s,x):
        x=x.mean(1,keepdim=True)
        Gx=F.conv3d(x,s.sobel_x,padding=1); Gy=F.conv3d(x,s.sobel_y,padding=1); Gz=F.conv3d(x,s.sobel_z,padding=1)
        return torch.sqrt(Gx**2+Gy**2+Gz**2+1e-8)
class _ResidualEdgeBlock3D(nn.Module):
    def __init__(s,ci,co,dropout=0.2):
        super().__init__(); s.conv1=nn.Conv3d(ci,co,3,padding=1,bias=False); s.bn1=_gn(co)
        s.conv2=nn.Conv3d(co,co,3,padding=1,bias=False); s.bn2=_gn(co); s.edge=_MultiScaleEdgeDetector3D(ci)
        s.attn=nn.Sequential(nn.Conv3d(co+1,co,3,padding=1,bias=False),_gn(co),nn.Sigmoid())
        s.res=nn.Conv3d(ci,co,1,bias=False) if ci!=co else nn.Identity(); s.dropout=nn.Dropout3d(dropout)
    def forward(s,x):
        idt=s.res(x); out=F.relu(s.bn1(s.conv1(x))); out=s.dropout(F.relu(s.bn2(s.conv2(out)))); edge=s.edge(x)
        if edge.shape[2:]!=out.shape[2:]: edge=F.interpolate(edge,size=out.shape[2:],mode='trilinear',align_corners=False)
        attn=s.attn(torch.cat([out,edge],1)); return F.relu(out*attn+idt)
class _TransformerBottleneck3D(nn.Module):
    def __init__(s,dim,num_heads=4,dropout=0.2):
        super().__init__(); s.dim=dim; s.norm1=nn.LayerNorm(dim)
        s.attn=nn.MultiheadAttention(dim,num_heads,dropout=dropout); s.norm2=nn.LayerNorm(dim)
        s.ff=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(s,x):
        B,C,D,H,W=x.shape; seq=x.flatten(2).permute(2,0,1); sn=s.norm1(seq)
        ao,_=s.attn(sn,sn,sn); seq=seq+ao; seq=seq+s.ff(s.norm2(seq))
        return seq.permute(1,2,0).view(B,C,D,H,W)
class _UpConv3D(nn.Module):
    def __init__(s,ci,co,block='double_conv',dr=0.2):
        super().__init__(); s.up=nn.Upsample(scale_factor=2,mode='trilinear',align_corners=False)
        s.conv={'double_conv':_DoubleConv3D,'res_conv':_ResidualConvBlock3D,'res_edge':_ResidualEdgeBlock3D}[block](ci,co,dr)
    def forward(s,x,skip):
        x=s.up(x)
        if x.shape[2:]!=skip.shape[2:]: skip=F.interpolate(skip,size=x.shape[2:],mode='trilinear',align_corners=False)
        return s.conv(torch.cat([skip,x],1))
class edge_att(nn.Module):
    def __init__(self,in_ch=1,out_ch=1,base_ch=16,dropout_rate=0.2):
        super().__init__()
        self.enc1=_ResidualEdgeBlock3D(in_ch,base_ch,dropout_rate); self.pool1=nn.MaxPool3d(2)
        self.enc2=_ResidualEdgeBlock3D(base_ch,base_ch*2,dropout_rate); self.pool2=nn.MaxPool3d(2)
        self.enc3=_ResidualEdgeBlock3D(base_ch*2,base_ch*4,dropout_rate); self.pool3=nn.MaxPool3d(2)
        self.enc4=_ResidualEdgeBlock3D(base_ch*4,base_ch*8,dropout_rate); self.pool4=nn.MaxPool3d(2)
        self.bottleneck_prep=nn.Conv3d(base_ch*8,base_ch*8,1); self.bottleneck_norm=_gn(base_ch*8)
        self.bottleneck_transformer=_TransformerBottleneck3D(base_ch*8); self.bottleneck_post=nn.Conv3d(base_ch*8,base_ch*8,1)
        self.up4=_UpConv3D(base_ch*8+base_ch*8,base_ch*8,'res_conv',dropout_rate)
        self.up3=_UpConv3D(base_ch*8+base_ch*4,base_ch*4,'double_conv',dropout_rate)
        self.up2=_UpConv3D(base_ch*4+base_ch*2,base_ch*2,'double_conv',dropout_rate)
        self.up1=_UpConv3D(base_ch*2+base_ch,base_ch,'double_conv',dropout_rate)
        self.out_conv=nn.Conv3d(base_ch,out_ch,1)
    def forward(self,x):
        s1=self.enc1(x); s2=self.enc2(self.pool1(s1)); s3=self.enc3(self.pool2(s2)); s4=self.enc4(self.pool3(s3))
        b=self.pool4(s4); b=F.relu(self.bottleneck_norm(self.bottleneck_prep(b)))
        b=self.bottleneck_transformer(b); b=self.bottleneck_post(b)
        d4=self.up4(b,s4); d3=self.up3(d4,s3); d2=self.up2(d3,s2); d1=self.up1(d2,s1)
        return self.out_conv(d1)

def build_model(arch, cfg, n_classes):
    """baseline/sea/fresnel share the backbone (modules at decoder); edgeatt is the Paper-1 reference model."""
    if arch in ("edgeatt","edge_att"):
        return edge_att(in_ch=1, out_ch=n_classes, base_ch=cfg.base_channels)
    return CompareUNet3D(cfg, n_classes, arch)


## 5. Losses and metrics

In [ ]:
import torch.nn.functional as F
from scipy.ndimage import distance_transform_edt, binary_erosion, gaussian_filter

def dice_loss_mc(logits,t,n,eps=1.):
    p=torch.softmax(logits,1); oh=F.one_hot(t,n).permute(0,4,1,2,3).float()
    num=2*(p[:,1:]*oh[:,1:]).sum((2,3,4))+eps; den=p[:,1:].sum((2,3,4))+oh[:,1:].sum((2,3,4))+eps
    return (1-num/den).mean()
def total_loss(logits,t,cfg,n): return cfg.w_dice*dice_loss_mc(logits,t,n)+cfg.w_ce*F.cross_entropy(logits,t)
def dice_bin(p,t,eps=1e-6): p=p.astype(bool);t=t.astype(bool); return float((2*(p&t).sum()+eps)/(p.sum()+t.sum()+eps))
def hd95_bin(p,t):
    p=p.astype(bool);t=t.astype(bool)
    if p.sum()==0 or t.sum()==0: return float("nan")
    dt_t=distance_transform_edt(~t);dt_p=distance_transform_edt(~p)
    sp=p&~binary_erosion(p,border_value=0); st=t&~binary_erosion(t,border_value=0)
    d=np.concatenate([dt_t[sp],dt_p[st]]); return float(np.percentile(d,95)) if d.size else float("nan")
def per_class(pred,t,n): return {c:(dice_bin(pred==c,t==c),hd95_bin(pred==c,t==c)) for c in range(1,n)}
def meanfg(pred,t,n): return float(np.mean([dice_bin(pred==c,t==c) for c in range(1,n)]))


## 6. Train all three arms

In [ ]:
from torch.utils.data import DataLoader
import pandas as pd, json

def _loader(cases, aug, bs):
    return DataLoader(DS(cases,cfg,LMAP,aug=aug), batch_size=bs, shuffle=aug, num_workers=0)

def train_one(arch, tr_c, val_c, ckpt_path, hist_path, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    net=build_model(arch,cfg,NCLS).to(cfg.device)
    opt=torch.optim.Adam(net.parameters(),cfg.lr,weight_decay=cfg.weight_decay)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,cfg.epochs)
    scaler=torch.amp.GradScaler('cuda',enabled=cfg.amp and cfg.device=="cuda")
    tr=_loader(tr_c,True,cfg.batch_size); va=_loader(val_c,False,1)
    def vdice():
        net.eval(); ds=[]
        with torch.no_grad():
            for x,y,_ in va: ds.append(meanfg(net(x.to(cfg.device)).argmax(1).cpu().numpy()[0],y.numpy()[0],NCLS))
        return float(np.mean(ds))
    hist=[]; best=-1; bs=None; since=0
    for ep in range(cfg.epochs):
        net.train(); ls=[]
        for x,y,_ in tr:
            x,y=x.to(cfg.device),y.to(cfg.device); opt.zero_grad()
            with torch.amp.autocast('cuda',enabled=cfg.amp and cfg.device=="cuda"): loss=total_loss(net(x),y,cfg,NCLS)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); ls.append(loss.item())
        sch.step(); vd=vdice(); hist.append(vd)
        if vd>best+1e-4: best=vd; bs={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}; since=0
        else: since+=1
        if ep%10==0 or ep==cfg.epochs-1: print(f"      ep {ep:3d} loss {np.mean(ls):.4f} val {vd:.4f}")
        if cfg.early_stop_patience and since>=cfg.early_stop_patience:
            print(f"      early stop @ ep {ep}"); break
    net.load_state_dict(bs); torch.save(bs,ckpt_path); json.dump(hist,open(hist_path,"w"))
    return net

@torch.no_grad()
def eval_seed(net, arch, seed, out_csv):
    net.eval(); rows=[]
    for ip,lp,cid in test_all:
        img,lab=prep(ip,lp,cfg,LMAP); pr=net(torch.from_numpy(img)[None,None].to(cfg.device)).argmax(1).cpu().numpy()[0]
        for c,(dc,hd) in per_class(pr,lab,NCLS).items():
            rows.append(dict(arch=arch,seed=seed,case_id=cid,klass=c,klass_name=CLASS_NAMES.get(c,f"class{c}"),dice=dc,hd95=hd))
    pd.DataFrame(rows).to_csv(out_csv,index=False)

# ---- multi-seed loop, resumable per (arch, seed) ----
print(f"multi-seed run: arches={cfg.arches} x seeds={cfg.seeds}  (fixed test={len(test_all)})")
for seed in cfg.seeds:
    tr_c,val_c=split_trainval(seed)
    for arch in cfg.arches:
        tag=f"{arch}_seed{seed}"; ckpt=CKPT/f"{tag}.pt"; hist=CKPT/f"{tag}_hist.json"; pcsv=TBL/f"per_case_{tag}.csv"
        if ckpt.exists() and pcsv.exists() and not cfg.retrain:
            print(f"== {tag}: done -> skip =="); continue
        print(f"== {tag}  (train={len(tr_c)} val={len(val_c)}) ==")
        net=train_one(arch,tr_c,val_c,ckpt,hist,seed)
        eval_seed(net,arch,seed,pcsv)
        del net
        if cfg.device=="cuda": torch.cuda.empty_cache()
        print(f"   {tag} -> {pcsv.name}")
print("multi-seed training complete.")


## 7. Per-class test evaluation + robustness

In [ ]:
import pandas as pd, json
from scipy import stats

fs=sorted(TBL.glob("per_case_*_seed*.csv"))
alld=pd.concat([pd.read_csv(f) for f in fs],ignore_index=True)
arches=[a for a in cfg.arches if a in alld.arch.unique()]
seeds_done=[int(x) for x in sorted(alld.seed.unique())]
print(f"pooled {len(fs)} files | arches={arches} | seeds={seeds_done} | test cases={alld.case_id.nunique()}")

# per (arch,class): grand mean Dice + std ACROSS seed-level means
rows=[]
for (a,kl),g in alld.groupby(["arch","klass_name"]):
    sm=g.groupby("seed").dice.mean()
    rows.append(dict(arch=a,klass=kl,dice_mean=round(g.dice.mean(),4),seed_std=round(float(sm.std()),4),n_seeds=g.seed.nunique()))
summary=pd.DataFrame(rows).sort_values(["klass","arch"]).reset_index(drop=True)

# PRIMARY: per-case mean Dice across seeds, paired Wilcoxon vs baseline (per class)
sig=[]
for kl in alld.klass_name.unique():
    pm=alld[alld.klass_name==kl].groupby(["arch","case_id"]).dice.mean().unstack(0)
    if "baseline" not in pm.columns: continue
    for a in [c for c in pm.columns if c!="baseline"]:
        x=pm[a]; b=pm["baseline"]; d=(x-b).dropna()
        p=float(stats.wilcoxon(x,b).pvalue) if len(d)>1 else float("nan")
        sig.append(dict(klass=kl,arch=a,mean_delta=round(float(d.mean()),4),
                        median_arm=round(float(x.median()),3),median_base=round(float(b.median()),3),
                        win=int((d>0).sum()),n=int(len(d)),p_value=round(p,4)))
sig=pd.DataFrame(sig)
# Holm correction across all primary tests
if not sig.empty:
    order=sig.p_value.fillna(1).sort_values().index; m=len(order)
    adj={}
    for rank,idx in enumerate(order): adj[idx]=min(1.0,(sig.loc[idx,"p_value"] if sig.loc[idx,"p_value"]==sig.loc[idx,"p_value"] else 1.0)*(m-rank))
    sig["holm_p"]=[round(adj[i],4) for i in sig.index]
    sig["sig"]=sig["holm_p"]<0.05

# per-SEED delta (arm - baseline) mean Dice, per class -> consistency of sign
seed_delta=[]
for kl in alld.klass_name.unique():
    for a in [x for x in arches if x!="baseline"]:
        for s in seeds_done:
            ga=alld[(alld.arch==a)&(alld.seed==s)&(alld.klass_name==kl)].set_index("case_id").dice
            gb=alld[(alld.arch=="baseline")&(alld.seed==s)&(alld.klass_name==kl)].set_index("case_id").dice
            common=ga.index.intersection(gb.index)
            seed_delta.append(dict(klass=kl,arch=a,seed=s,delta=round(float((ga[common]-gb[common]).mean()),4)))
seed_delta=pd.DataFrame(seed_delta)

per_case_pooled=alld
print("\nPER-CLASS (mean over seeds&cases | std across seeds):"); print(summary.to_string(index=False))
if not sig.empty: print("\nPRIMARY paired test (seed-averaged per-case Dice, Wilcoxon vs baseline):"); print(sig.to_string(index=False))
if not seed_delta.empty:
    print("\nPER-SEED delta vs baseline (sign consistency):")
    print(seed_delta.pivot_table(index=["klass","arch"],columns="seed",values="delta").to_string())


## 8. Figures

In [ ]:
import matplotlib.pyplot as plt, numpy as np
COL={"baseline":"#999999","sea":"#ee6677","fresnel":"#4477aa"}
classes=sorted(summary.klass.unique())
# Fig1: per-class mean Dice +/- seed std
fig,ax=plt.subplots(figsize=(7,4)); x=np.arange(len(classes)); w=0.8/max(1,len(arches))
for i,a in enumerate(arches):
    means=[summary[(summary.arch==a)&(summary.klass==k)].dice_mean.values[0] if len(summary[(summary.arch==a)&(summary.klass==k)]) else 0 for k in classes]
    errs=[summary[(summary.arch==a)&(summary.klass==k)].seed_std.values[0] if len(summary[(summary.arch==a)&(summary.klass==k)]) else 0 for k in classes]
    ax.bar(x+(i-(len(arches)-1)/2)*w,means,w,yerr=errs,capsize=3,label=a,color=COL.get(a,"#888"))
ax.set_xticks(x); ax.set_xticklabels(classes); ax.set_ylabel("Dice (mean ± seed std)"); ax.set_title(f"Per-class Dice over {len(seeds_done)} seeds"); ax.legend()
plt.tight_layout(); fig.savefig(FIG_:=FIG if False else (OUT/'Figures'/'fig1_per_class_mean_std.png'),dpi=160,bbox_inches="tight"); plt.show()
# Fig2: per-seed delta strip (consistency)
if not seed_delta.empty:
    fig,ax=plt.subplots(figsize=(7,4))
    for kl in classes:
        for a in [x for x in arches if x!="baseline"]:
            sd=seed_delta[(seed_delta.klass==kl)&(seed_delta.arch==a)]
            ax.scatter([f"{kl}\n{a}"]*len(sd),sd.delta,s=40)
    ax.axhline(0,color="k",lw=0.8); ax.set_ylabel("Δ Dice vs baseline (per seed)"); ax.set_title("Per-seed effect (above 0 = better than baseline)")
    plt.tight_layout(); fig.savefig(OUT/'Figures'/'fig2_per_seed_delta.png',dpi=160,bbox_inches="tight"); plt.show()
print("figures saved")


## 9. Tables + summary

In [ ]:
import json
def _jsonable(o):
    if isinstance(o,(np.integer,)): return int(o)
    if isinstance(o,(np.floating,)): return float(o)
    if isinstance(o,(np.bool_,)): return bool(o)
    return str(o)
per_case_pooled.round(5).to_csv(TBL/"per_case_pooled.csv",index=False)
summary.to_csv(TBL/"per_class_summary.csv",index=False)
if not sig.empty: sig.to_csv(TBL/"significance.csv",index=False)
if not seed_delta.empty: seed_delta.to_csv(TBL/"per_seed_delta.csv",index=False)
json.dump(dict(n_classes=NCLS,target_shape=list(cfg.target_shape),seeds=[int(x) for x in seeds_done],
              per_class=summary.to_dict("records"),
              significance=(sig.to_dict("records") if not sig.empty else []),
              per_seed_delta=(seed_delta.to_dict("records") if not seed_delta.empty else [])),
          open(TBL/"results_bundle.json","w"),indent=2,default=_jsonable)

L=[]; out=lambda s="":(L.append(s),print(s))
out("PANCREAS — MULTI-SEED confirmation (baseline vs Fresnel)"); out("="*70)
out(f"{NCLS}-way (pancreas/mass) @ {tuple(cfg.target_shape)} | fixed test={alld.case_id.nunique()} | "
    f"seeds={seeds_done} | arms={arches}")
out("\nPer-class Dice (mean over seeds&cases ± std across seeds):")
for kl in sorted(summary.klass.unique()):
    out(f"  [{kl}]")
    for a in arches:
        r=summary[(summary.arch==a)&(summary.klass==kl)]
        if len(r): out(f"     {a:<9} {r.dice_mean.values[0]:.4f} ± {r.seed_std.values[0]:.4f}")
if not sig.empty:
    out("\nPRIMARY test — seed-averaged per-case Dice, paired Wilcoxon vs baseline:")
    for _,r in sig.iterrows():
        tag=" *SIG*" if r.get("sig",False) else ""
        out(f"   [{r.klass:<8}] {r.arch:<9} Δ={r.mean_delta:+.4f}  median {r.median_arm} vs {r.median_base}  "
            f"win {r.win}/{r.n}  p={r.p_value}  Holm={r.get('holm_p','-')}{tag}")
if not seed_delta.empty:
    out("\nPer-seed Δ vs baseline (consistency of sign):")
    pv=seed_delta.pivot_table(index=["klass","arch"],columns="seed",values="delta")
    for (kl,a),row in pv.iterrows():
        vals=[row[s] for s in pv.columns]; pos=sum(v>0 for v in vals)
        out(f"   [{kl:<8}] {a:<9} per-seed Δ={[round(v,3) for v in vals]}  ({pos}/{len(vals)} positive)")
out("\nReading: PRIMARY test averages out init/split noise (high power). A *SIG* on mass = the")
out("Fresnel mass effect is real and survives multiple-comparison correction across seeds.")
out("Per-seed consistency (n/n positive) shows whether the direction holds run-to-run.")
open(SUMMARY,"w").write("\n".join(L)+"\n")
print("\nsummary ->",SUMMARY)


---
**Notes**
- Restore the original Decathlon `{0,1,2}` labels (nearest-neighbor through your preprocessing, no
  binarization) so this runs the real 3-way anterior/posterior task; otherwise it falls back to binary.
- To add your Paper-1 **EdgeAttResUNet3D** as a fourth arm, drop its module into `make_module` (or its
  full model) and append `"edgeatt"` to `cfg.arches`.
- `cfg.data_root` points at the folder holding `train/` and `test/`.
